# DraxNet YOLO26 Comparison Notebook

This notebook trains and tests the professor's DraxNet YOLO26 model on the SkyFusion dataset for comparison against the custom SkyFusion YOLO26 model.

It uses a Google Drive dataset path by default to avoid Colab RAM crashes from `files.upload()`.

In [ ]:
!pip -q install ultralytics pyyaml

import os
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

ROOT = Path('/content/draxnet_comparison')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Dataset

Put the SkyFusion dataset zip in Google Drive, then update `DATASET_ZIP` if your filename is different. The zip should contain `data.yaml`, `train`, `valid`, and `test`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ZIP = Path('/content/drive/MyDrive/skyfusion.v1i.yolov11.zip')
assert DATASET_ZIP.exists(), f'Update DATASET_ZIP. File not found: {DATASET_ZIP}'

DATASET_DIR = ROOT / 'skyfusion_dataset'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
!unzip -q -o "{DATASET_ZIP}" -d "{DATASET_DIR}"

yaml_files = list(DATASET_DIR.rglob('data.yaml'))
assert yaml_files, 'No data.yaml found. The zip should include data.yaml plus train/valid/test folders.'
DATA_YAML = yaml_files[0]

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = DATA_YAML.parent
data['path'] = str(dataset_root)
data['train'] = str((dataset_root / 'train' / 'images').resolve())
data['val'] = str((dataset_root / 'valid' / 'images').resolve())
data['test'] = str((dataset_root / 'test' / 'images').resolve())
data['nc'] = 3
data['names'] = ['Aircraft', 'ship', 'vehicle']

COLAB_DATA_YAML = ROOT / 'skyfusion_data.yaml'
with open(COLAB_DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(COLAB_DATA_YAML)
print(yaml.safe_dump(data, sort_keys=False))

## Professor DraxNet YOLO26

This cell registers DraxNet and writes the DraxNet YOLO26 YAML. The architecture matches the local `draxnet-yolo26.yaml`: DraxNet backbone, P3/P4/P5 detection, `end2end=True`, and `reg_max=1`.

In [ ]:
from typing import Sequence

import torch.nn as nn
from ultralytics.nn.modules import Conv
import ultralytics.nn.tasks as tasks


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class LayerNorm2D(nn.Module):
    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(num_channels, eps=eps)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2)


class SelfAttention2D(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_dropout=0.0, proj_dropout=0.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.norm = nn.GroupNorm(1, dim)
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=qkv_bias)
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)
        self.scale = self.head_dim ** -0.5
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x):
        batch_size, channels, height, width = x.shape
        residual = x
        qkv = self.qkv(self.norm(x))
        q, k, v = torch.chunk(qkv, 3, dim=1)
        q = q.reshape(batch_size, self.num_heads, self.head_dim, height * width).transpose(-2, -1)
        k = k.reshape(batch_size, self.num_heads, self.head_dim, height * width)
        v = v.reshape(batch_size, self.num_heads, self.head_dim, height * width).transpose(-2, -1)
        attn = torch.matmul(q, k) * self.scale
        attn = self.attn_dropout(attn.softmax(dim=-1))
        out = torch.matmul(attn, v)
        out = out.transpose(-2, -1).contiguous().reshape(batch_size, channels, height, width)
        return residual + self.proj_dropout(self.proj(out))


class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, expansion=4, kernel_size=7, layer_scale_init_value=1e-6, dropout=0.0):
        super().__init__()
        hidden_dim = dim * expansion
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim)
        self.norm = LayerNorm2D(dim)
        self.pwconv1 = nn.Conv2d(dim, hidden_dim, kernel_size=1)
        self.activation = nn.GELU()
        self.pwconv2 = nn.Conv2d(hidden_dim, dim, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.layer_scale = nn.Parameter(layer_scale_init_value * torch.ones(dim)) if layer_scale_init_value > 0 else None

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.activation(x)
        x = self.pwconv2(x)
        if self.layer_scale is not None:
            x = x * self.layer_scale.view(1, -1, 1, 1)
        return residual + self.dropout(x)


def _resolve_num_heads(dim, max_heads=8):
    for num_heads in range(min(max_heads, dim), 0, -1):
        if dim % num_heads == 0:
            return num_heads
    return 1


def _resolve_efficient_dim(dim):
    reduced_dim = max(32, dim // 2)
    while reduced_dim > 1 and dim % reduced_dim != 0:
        reduced_dim -= 1
    return reduced_dim


class DraxBlock(nn.Module):
    def __init__(self, dim=128, use_attention=True, efficient=True, drop_path=0.0):
        super().__init__()
        self.use_attention = use_attention
        self.efficient = efficient
        self.convnext = ConvNeXtBlock(dim)
        self.drop_path = DropPath(drop_path)
        if not use_attention:
            self.attention = None
            self.attn_down = None
            self.attn_up = None
            return
        attention_dim = _resolve_efficient_dim(dim) if efficient else dim
        self.attention = SelfAttention2D(attention_dim, num_heads=_resolve_num_heads(attention_dim))
        self.attn_down = nn.Conv2d(dim, attention_dim, kernel_size=1) if efficient else None
        self.attn_up = nn.Conv2d(attention_dim, dim, kernel_size=1) if efficient else None

    def forward(self, x):
        conv_delta = self.convnext(x) - x
        if not self.use_attention or self.attention is None:
            return x + self.drop_path(conv_delta)
        if self.efficient:
            reduced = self.attn_down(x)
            attention_delta = self.attn_up(self.attention(reduced) - reduced)
        else:
            attention_delta = self.attention(x) - x
        return x + self.drop_path(0.5 * (conv_delta + attention_delta))


class BasicResidualBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        if self.downsample is not None:
            identity = self.downsample(identity)
        return self.relu(x + identity)


class DraxResidualBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, use_attention=True, efficient_attention=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.drax = DraxBlock(dim=out_channels, use_attention=use_attention, efficient=efficient_attention)
        self.proj = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.proj(self.drax(x)))
        if self.downsample is not None:
            identity = self.downsample(identity)
        return self.relu(x + identity)


class DraxNet(nn.Module):
    def __init__(self, *args, c1=3, c2=1024, layers=(2, 2, 2, 2), stage_block_types=('basic', 'basic', 'basic', 'drax'), use_attention=True, efficient_attention=True, out_channels=(256, 512, 1024), zero_init_residual=False):
        super().__init__()
        if args:
            if len(args) >= 7:
                c1, c2, layers, stage_block_types, use_attention, efficient_attention, out_channels = args[:7]
            else:
                c2, layers, stage_block_types, use_attention, efficient_attention, out_channels = args[:6]
        self.c2 = c2
        self.inplanes = 64
        self.use_attention = use_attention
        self.efficient_attention = efficient_attention
        self.conv1 = nn.Conv2d(int(c1), self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        stage_blocks = [self._resolve_stage_block(block_type) for block_type in stage_block_types]
        self.layer1 = self._make_layer(stage_blocks[0], 64, layers[0])
        self.layer2 = self._make_layer(stage_blocks[1], 128, layers[1], stride=2)
        self.layer3 = self._make_layer(stage_blocks[2], 256, layers[2], stride=2)
        self.layer4 = self._make_layer(stage_blocks[3], 512, layers[3], stride=2)
        self.p3_proj = Conv(128, out_channels[0], k=1, s=1)
        self.p4_proj = Conv(256, out_channels[1], k=1, s=1)
        self.p5_proj = Conv(512, out_channels[2], k=1, s=1)
        self._init_weights()
        if zero_init_residual:
            for module in self.modules():
                if isinstance(module, (BasicResidualBlock, DraxResidualBlock)):
                    nn.init.constant_(module.bn2.weight, 0)

    def _resolve_stage_block(self, block_type):
        normalized = str(block_type).strip().lower()
        if normalized == 'basic':
            return BasicResidualBlock
        if normalized in {'cax', 'drax'}:
            return DraxResidualBlock
        raise ValueError(f'Unsupported DraxNet stage block type: {block_type}')

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(planes * block.expansion))
        layers = [block(self.inplanes, planes, stride=stride, downsample=downsample, use_attention=self.use_attention, efficient_attention=self.efficient_attention) if block is DraxResidualBlock else block(self.inplanes, planes, stride=stride, downsample=downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, use_attention=self.use_attention, efficient_attention=self.efficient_attention) if block is DraxResidualBlock else block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        p3 = self.layer2(x)
        p4 = self.layer3(p3)
        p5 = self.layer4(p4)
        return [self.p3_proj(p3), self.p4_proj(p4), self.p5_proj(p5)]


tasks.DraxNet = DraxNet
print('DraxNet registered.')

In [ ]:
MODEL_YAML = ROOT / 'draxnet-yolo26.yaml'
MODEL_YAML.write_text('''
nc: 3
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, DraxNet, [1024, [2, 2, 2, 2], [basic, basic, basic, drax], True, True, [256, 512, 1024]]]
  - [0, 1, Index, [256, 0]]
  - [0, 1, Index, [512, 1]]
  - [0, 1, Index, [1024, 2]]
  - [-1, 1, SPPF, [1024, 5, 3, True]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 1], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 1, C3k2, [1024, True, 0.5, True]]
  - [[11, 14, 17], 1, Detect, [nc]]
'''.strip() + '\n')

model = YOLO(str(MODEL_YAML))
model.info(detailed=False)
print(MODEL_YAML)

## Train DraxNet

These defaults are intentionally low-RAM for Colab. Increase `BATCH` to 4 or 8 only if your runtime has enough memory.

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

EPOCHS = 150
BATCH = 2
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(MODEL_YAML))
results = model.train(
    data=str(COLAB_DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    project=str(ROOT / 'runs' / 'final_project'),
    name='draxnet_yolo26',
    exist_ok=True,
    pretrained=False,
    plots=False,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.0015,
    lrf=0.02,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=50,
    close_mosaic=20,
    mosaic=1.0,
    mixup=0.05,
    cutmix=0.05,
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.35,
    degrees=7.0,
    translate=0.12,
    scale=0.55,
    shear=2.0,
    fliplr=0.5,
    flipud=0.5,
    cls_pw=0.25,
    deterministic=False,
    amp=torch.cuda.is_available(),
    cache=False,
)

BEST_PT = Path(model.trainer.save_dir) / 'weights' / 'best.pt'
print('Best checkpoint:', BEST_PT)

## Test Split Results

Use this score as the DraxNet comparison point.

In [ ]:
metrics = YOLO(str(BEST_PT)).val(
    data=str(COLAB_DATA_YAML),
    split='test',
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    project=str(ROOT / 'runs' / 'final_project'),
    name='draxnet_yolo26_test',
    exist_ok=True,
    plots=True,
    save_json=True,
    conf=0.001,
    iou=0.70,
)

print(metrics.results_dict)
print(f'mAP50: {metrics.box.map50:.5f}')
print(f'mAP50-95: {metrics.box.map:.5f}')
print(f'precision: {metrics.box.mp:.5f}')
print(f'recall: {metrics.box.mr:.5f}')

In [ ]:
SUBMISSION_DIR = Path('/content/drive/MyDrive/skyfusion_draxnet_comparison')
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
!cp "{BEST_PT}" "{SUBMISSION_DIR / 'draxnet_best.pt'}"
!cp -r "{ROOT / 'runs' / 'final_project' / 'draxnet_yolo26_test'}" "{SUBMISSION_DIR / 'test_results'}"
print('Saved to:', SUBMISSION_DIR)